# Tarea 2 · Parte 2 — Evaluación (versión recortada)

¿Qué tan bien leyeron el OCR y el ASR de la Parte 1, y qué tan confiable es una extracción estructurada hecha por un LLM?

- **No se vuelve a correr ningún modelo de OCR ni ASR.** Todo sale de `datos/documentos.csv` y `datos/transcripciones.csv`.
- **CER** en 10 documentos: las 8 páginas del PDF del MINJUS y 2 transcritos a mano leyendo la imagen (un manuscrito y una foto difícil).
- **WER** en 10 clips de OpenSLR 73 con duraciones distintas, contra su transcripción oficial.
- **Extracción LLM** de los 95 textos y **auditoría** de 20 casos al azar con semilla fija.

Por tiempo de cómputo se omiten la comparación con modelos alternativos y los casos difíciles (ver *Limitaciones* al final).

## 1. Datos

In [1]:
import json
import re
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, r"C:\Users\Renat\Documentos\GitHub\Data-Science-Python\sessions\13-audio-a-texto")
from voz import cer, normalizar, wer

DATOS = Path("datos")
pd.set_option("display.max_colwidth", 80)

documentos = pd.read_csv(DATOS / "documentos.csv", keep_default_na=False)
transcripciones = pd.read_csv(DATOS / "transcripciones.csv", keep_default_na=False)
print(f"Ejecutado el {datetime.now():%Y-%m-%d %H:%M} · {len(documentos)} documentos · {len(transcripciones)} clips")

Ejecutado el 2026-10-26 00:06 · 35 documentos · 60 clips


## 2. CER en 10 documentos

`voz.cer()` normaliza ambos textos (minúsculas, sin puntuación ni espacios repetidos; las tildes se quedan) y cuenta las ediciones
de carácter necesarias para pasar del texto OCR a la referencia, divididas entre el largo de la referencia.

- **Páginas del PDF:** la referencia es la capa de texto del PDF extraída con `pypdf` (columna `texto_referencia`).
- **`man_03_pizarra_menu_1`** (manuscrito, tiza de colores) y **`foto_07_menu_las_lenas`** (foto con platos tachados a mano y
  bordes cortados): la referencia se transcribió a mano mirando la imagen, tal como está escrito, incluidas las palabras
  cortadas por el borde. Está en `datos/referencias_manuales.csv`.

In [2]:
manuales = pd.read_csv(DATOS / "referencias_manuales.csv").set_index("id").texto_referencia
evaluados = documentos[(documentos.texto_referencia != "") | documentos.id.isin(manuales.index)].copy()
evaluados["texto_referencia"] = evaluados.id.map(manuales).fillna(evaluados.texto_referencia)
evaluados["referencia"] = np.where(evaluados.id.isin(manuales.index), "a mano", "pypdf")
evaluados["caracteres_ref"] = evaluados.texto_referencia.map(lambda t: len(normalizar(t)))
evaluados["cer"] = [cer([r], [h]) for r, h in zip(evaluados.texto_referencia, evaluados.texto_ocr)]

tabla_cer = evaluados[["id", "tipo", "referencia", "caracteres_ref", "cer"]].sort_values("cer").reset_index(drop=True)
tabla_cer.to_csv(DATOS / "metricas_cer.csv", index=False)
print(f"CER mediano: {tabla_cer.cer.median():.1%} · rango {tabla_cer.cer.min():.1%}–{tabla_cer.cer.max():.1%}")
tabla_cer.style.format({"cer": "{:.1%}"})

CER mediano: 35.6% · rango 0.9%–54.2%


,id,tipo,referencia,caracteres_ref,cer
0,imp_01_resolucion_minjus_4378_2025_p04,impreso,pypdf,2937,0.9%
1,man_03_pizarra_menu_1,manuscrito,a mano,211,15.2%
2,foto_07_menu_las_lenas,foto,a mano,905,18.0%
3,imp_01_resolucion_minjus_4378_2025_p05,impreso,pypdf,3932,30.6%
4,imp_01_resolucion_minjus_4378_2025_p06,impreso,pypdf,3637,33.0%
5,imp_01_resolucion_minjus_4378_2025_p03,impreso,pypdf,3162,38.1%
6,imp_01_resolucion_minjus_4378_2025_p02,impreso,pypdf,2993,40.3%
7,imp_01_resolucion_minjus_4378_2025_p07,impreso,pypdf,3196,41.5%
8,imp_01_resolucion_minjus_4378_2025_p08,impreso,pypdf,3054,42.9%
9,imp_01_resolucion_minjus_4378_2025_p01,impreso,pypdf,2112,54.2%


**El CER de las páginas del PDF mide sobre todo orden, no letras.** La página 4 da menos de 1 %: ahí la capa de texto de `pypdf`
sigue el mismo orden que el OCR, y se ve que el reconocedor casi no se equivoca de carácter. En las otras siete el CER sube a 30–54 %
porque el sello lateral ("Esta es una copia auténtica imprimible…"), el encabezado y las notas al pie están en ambos textos pero en
otra posición, y el CER, que es una distancia de edición, cobra cada bloque desplazado como si se hubiera borrado y vuelto a escribir.
Quitando solo el sello de la referencia, la página 3 baja de 38 % a 24 %. Por eso el CER mediano exagera el error de lectura en
impresos; los dos documentos transcritos a mano (15 % el manuscrito, 18 % la foto) son el mejor indicio de errores reales de lectura.

## 3. WER en 10 clips de OpenSLR 73

Los 40 clips se ordenan por duración y se toman 10 equiespaciados, del más corto al más largo. `voz.wer()` normaliza igual que el
CER y cuenta palabras. Antes se quitan las marcas de hablante (`H1:`, `[H2]`) y los seudónimos, que no están en el audio; en estos
clips de un solo hablante no hay ninguna, pero el filtro deja el cálculo listo para los audios diarizados.

In [3]:
MARCAS = re.compile(r"\[?\bH\d+\]?:?|\[[A-Z]+_?\d*\]")
sin_marcas = lambda t: re.sub(r"\s+", " ", MARCAS.sub(" ", t)).strip()

openslr = transcripciones[transcripciones.id.str.match(r"pe[fm]_") & (transcripciones.texto_referencia != "")]
openslr = openslr.sort_values("duracion_seg").reset_index(drop=True)
muestra = openslr.iloc[np.linspace(0, len(openslr) - 1, 10).round().astype(int)].copy()
muestra["texto_asr"] = muestra.texto_asr.map(sin_marcas)
muestra["wer"] = [wer([r], [h]) for r, h in zip(muestra.texto_referencia, muestra.texto_asr)]

tabla_wer = muestra[["id", "duracion_seg", "texto_referencia", "texto_asr", "wer"]].reset_index(drop=True)
tabla_wer.to_csv(DATOS / "metricas_wer.csv", index=False)
wer_global = wer(muestra.texto_referencia.tolist(), muestra.texto_asr.tolist())
print(f"WER mediano por clip: {tabla_wer.wer.median():.1%} · WER global de los 10 clips: {wer_global:.1%}")
tabla_wer.style.format({"wer": "{:.1%}", "duracion_seg": "{:.1f}"})

WER mediano por clip: 0.0% · WER global de los 10 clips: 1.9%


,id,duracion_seg,texto_referencia,texto_asr,wer
0,pem_05223_00388851533,2.9,En su auto hay un botiquín de emergencia,En su auto hay un botiquín de emergencia.,0.0%
1,pem_08421_00388917999,4.6,¿Qué tipo de temas le gustaría escuchar?,¿Qué tipo de tema le gustaría escuchar?,14.3%
2,pef_01208_00204186196,5.3,Es muy fácil resguardar su hogar de los huracanes,Es muy fácil reguardar su hogar de los huracanes.,11.1%
3,pef_01523_01958041774,5.6,¿Cómo puedo registrarme para entrar a la universidad?,¿Cómo puedo registrarme para entrar a la universidad?,0.0%
4,pem_09697_00226758927,5.9,¿Es vuelo directo o hace escala en algún punto?,¿Es vuelo directo o hace escala en algún punto?,0.0%
5,pem_08421_01683925931,6.2,Lo interesante de su educación es que ella no quería ser actriz quería ser psicóloga,"Lo interesante de su educación es que ella no quería ser actriz, quería ser psicóloga",0.0%
6,pem_08784_00330246008,6.5,El otro día me comí una sopa de champiñones muy picante,El otro día me comí una sopa de champiñones muy picante.,0.0%
7,pef_03397_01880729482,6.9,Como vienen personas de diferentes partes del mundo necesito traductores,Como vienen personas de diferentes partes del mundo necesito traductores,0.0%
8,pef_09334_01616249143,9.3,Aquí encontré un sitio donde puedes leer sobre las diferencias entre los géneros del teatro.,Aquí encontré un sitio donde puedes leer sobre las diferencias entre los géneros del teatro.,0.0%
9,pem_07508_01796320827,14.2,¿Qué me recomendarías una pecera de agua salada o una de agua dulce?,"¿Qué me recomendarías, una pecera de agua salada o una de agua dulce?",0.0%


Solo 2 de los 10 clips tienen errores, de una palabra cada uno: "tema" por "temas" y "reguardar" por "resguardar". Con frases leídas
en estudio Whisper casi no falla, y `voz.normalizar` ya evita que la puntuación y las mayúsculas cuenten como error. Fuera de esta
muestra aparecen errores de formato que un WER más justo debería normalizar, como "10 horas 35 minutos" frente a "diez horas
treinta y cinco minutos".

## 4. Extracción estructurada con un LLM

El LLM es Claude (`claude-opus-5-5`), trabajando dentro de la misma sesión de Claude Code: leyó solo `texto_ocr` o `texto_asr`
de cada fila (no las imágenes ni los audios) y devolvió un JSON con estos campos, con `null` cuando el texto no alcanza:

| Campo | Documentos | Audios |
|---|---|---|
| `modalidad` | `documento` | `audio` |
| `tema` | sí | sí |
| `resumen` | una oración | una oración |
| `tipo_documento` | sí | — |
| `n_hablantes` | — | sí |

El resultado está en `datos/extraccion_llm.csv` (`id, modalidad, campos_json, modelo_llm, fecha`). Aquí se valida el esquema.

In [4]:
extraccion = pd.read_csv(DATOS / "extraccion_llm.csv")
campos = pd.json_normalize(extraccion.campos_json.map(json.loads))
campos.insert(0, "id", extraccion.id)

ESPERADOS = {"documento": {"modalidad", "tema", "resumen", "tipo_documento"},
             "audio": {"modalidad", "tema", "resumen", "n_hablantes"}}
claves = extraccion.campos_json.map(lambda j: set(json.loads(j)))
assert all(c == ESPERADOS[m] for c, m in zip(claves, extraccion.modalidad))
assert (campos.modalidad == extraccion.modalidad).all()
assert set(extraccion.id) == set(documentos.id) | set(transcripciones.id)

print(f"{len(extraccion)} filas · modelo: {extraccion.modelo_llm.iloc[0]} · fecha: {extraccion.fecha.iloc[0]}")
campos.drop(columns="id").isna().sum().rename("nulos").to_frame().T

95 filas · modelo: claude-opus-5-5 (Claude Code, extracción hecha en la sesión, sin API) · fecha: 2026-10-26


,modalidad,tema,resumen,tipo_documento,n_hablantes
nulos,0,1,0,60,37


In [5]:
campos.groupby("modalidad").head(3)

,id,modalidad,tema,resumen,tipo_documento,n_hablantes
0,imp_01_resolucion_minjus_4378_2025_p01,documento,protección de datos personales,Antecedentes del procedimiento sancionador de la DGTAIPD contra Banco Pichin...,resolución administrativa,NaN
1,imp_01_resolucion_minjus_4378_2025_p02,documento,protección de datos personales,La DFI pide al banco explicar de dónde obtuvo el teléfono del denunciante y ...,resolución administrativa,NaN
2,imp_01_resolucion_minjus_4378_2025_p03,documento,protección de datos personales,Informe de fiscalización e inicio del procedimiento sancionador por tratar d...,resolución administrativa,NaN
35,aud_01_reunion_laboral,audio,monitoreo de un servicio de scraping de noticias,Una persona explica dos tipos de fallas del scraping de medios (estados vací...,NaN,2.0
36,agustino_1,audio,fútbol,Entrevista en la calle a una vendedora sobre el partido de Perú: espera que ...,NaN,2.0
37,agustino_2,audio,fútbol,Entrevista a dos vendedoras sobre el partido de Perú: destacan a Lapadula y ...,NaN,3.0


## 5. Auditoría de 20 casos

Muestra al azar con `random_state=2026`. Cada caso se revisó contra el texto fuente y, cuando hacía falta, contra la imagen o los
metadatos (`atribucion.csv`, `entrevista_simulada.csv`). La revisión está **prellenada por el mismo modelo que hizo la extracción**:
es un primer filtro, no una validación independiente, y queda pendiente que una persona la confirme en `datos/auditoria_llm.csv`.

In [6]:
auditoria = pd.read_csv(DATOS / "auditoria_llm.csv")
assert auditoria.id.tolist() == extraccion.sample(n=20, random_state=2026).id.tolist()

por_campo = auditoria[["tema_ok", "resumen_ok", "especifico_ok"]].mean()
acuerdo = auditoria[["tema_ok", "resumen_ok", "especifico_ok"]].to_numpy().mean()
print(f"Casos con todos los campos correctos: {auditoria.correcta.mean():.0%} ({auditoria.correcta.sum()} de {len(auditoria)})")
print(f"Acuerdo LLM-humano por campo: {acuerdo:.1%} · " + " · ".join(f"{c} {v:.0%}" for c, v in por_campo.items()))

Casos con todos los campos correctos: 90% (18 de 20)
Acuerdo LLM-humano por campo: 96.7% · tema_ok 100% · resumen_ok 95% · especifico_ok 95%


In [7]:
errores = auditoria[~auditoria.correcta]
for r in errores.itertuples():
    print(f"■ {r.id}\n  LLM: {r.campos_json}\n  Revisión: {r.comentario}\n")

■ imp_02_diploma_pucp
  LLM: {"modalidad": "documento", "tema": "educación superior", "resumen": "La PUCP confiere el grado de Bachiller en Ciencias Sociales con mención en Economía, otorgado el 11 de abril de 2025.", "tipo_documento": "diploma universitario"}
  Revisión: El 11 de abril de 2025 es la fecha en que optó al grado; el diploma se expidió el 14 de mayo de 2025. El resumen confunde ambas fechas.

■ entrevista_simulada
  LLM: {"modalidad": "audio", "tema": null, "resumen": "Secuencia de frases cortas sin hilo común: una granizada, la calma, la sal, el Amazonas y un mural de Diego Rivera.", "n_hablantes": null}
  Revisión: Son 2 hablantes (entrevistado y entrevistadora, según entrevista_simulada.csv). El texto_asr plano no tiene marcas de turno y el LLM devolvió null.



Los dos errores tienen causas distintas. En el diploma, el texto OCR es correcto y el error es del LLM, que confunde la fecha en que
el estudiante optó al grado con la de expedición del diploma. En la entrevista simulada, el LLM hace bien en no inventar, pero el
`null` es un error de cobertura: la Parte 1 guardó `texto_asr` sin marcas de turno, y desde el texto plano no se puede contar a los
hablantes. Pasarle el `.txt` diarizado de `datos/asr/` lo resolvería.

## 6. Tabla final

Los tiempos de la Parte 1 se leen de los outputs guardados en `01_extraccion.ipynb`, no se vuelven a medir.

In [8]:
salidas = " ".join(o.get("text", "") if isinstance(o.get("text", ""), str) else "".join(o["text"])
                   for c in json.loads(Path("01_extraccion.ipynb").read_text(encoding="utf-8"))["cells"]
                   if c["cell_type"] == "code" for o in c.get("outputs", []) if o["output_type"] == "stream")
min_ocr = float(re.search(r"(\d+) documentos leídos en ([\d.]+) min", salidas).group(2))
n_docs = int(re.search(r"(\d+) documentos leídos en", salidas).group(1))
n_clips, min_audio, min_asr = re.search(r"(\d+) clips · ([\d.]+) min de audio transcritos en ([\d.]+) min", salidas).groups()

resumen = pd.DataFrame([
    ("CER mediano (10 documentos)", f"{tabla_cer.cer.median():.1%}"),
    ("CER transcritos a mano (manuscrito · foto)",
     " · ".join(f"{tabla_cer.set_index('id').cer[i]:.1%}" for i in ["man_03_pizarra_menu_1", "foto_07_menu_las_lenas"])),
    ("WER mediano (10 clips OpenSLR)", f"{tabla_wer.wer.median():.1%}"),
    ("WER global (10 clips OpenSLR)", f"{wer_global:.1%}"),
    ("Acuerdo LLM-humano por campo (20 casos)", f"{acuerdo:.1%}"),
    ("Casos LLM con todos los campos correctos", f"{auditoria.correcta.mean():.0%}"),
    ("Parte 1 · OCR", f"{min_ocr:.1f} min para {n_docs} documentos ({60 * min_ocr / n_docs:.0f} s por documento)"),
    ("Parte 1 · ASR + diarización", f"{float(min_asr):.1f} min para {n_clips} clips y {float(min_audio):.1f} min de audio "
                                    f"({float(min_asr) / float(min_audio):.2f}× tiempo real)"),
    ("Hardware", "Todo en CPU, sin GPU: Intel Core i7-1355U, 16 GB de RAM"),
], columns=["métrica", "valor"])
resumen.to_csv(DATOS / "resumen_evaluacion.csv", index=False)
resumen

,métrica,valor
0,CER mediano (10 documentos),35.6%
1,CER transcritos a mano (manuscrito · foto),15.2% · 18.0%
2,WER mediano (10 clips OpenSLR),0.0%
3,WER global (10 clips OpenSLR),1.9%
4,Acuerdo LLM-humano por campo (20 casos),96.7%
5,Casos LLM con todos los campos correctos,90%
6,Parte 1 · OCR,19.7 min para 36 documentos (33 s por documento)
7,Parte 1 · ASR + diarización,48.5 min para 60 clips y 40.9 min de audio (1.19× tiempo real)
8,Hardware,"Todo en CPU, sin GPU: Intel Core i7-1355U, 16 GB de RAM"


## Limitaciones

1. **Recorte por tiempo de cómputo.** No se compararon modelos alternativos (PaddleOCR-VL para OCR; Qwen3-ASR, Parakeet o MMS para
   ASR) ni se armó el set de casos difíciles. En CPU, la sola Parte 1 tomó más de una hora; repetirla con otros modelos no cabía en el plazo.
2. **Referencias pocas y desiguales.** El CER de documentos descansa en 8 páginas de un mismo PDF digital (referencia `pypdf`, con
   su propio desorden) y solo 2 transcripciones a mano. El WER usa frases leídas en estudio: es el caso fácil del ASR. Las entrevistas
   reales, el quechua y la reunión de trabajo no tienen referencia, y es donde Whisper entra en bucles (Parte 1, sección 3).
3. **Auditoría no independiente.** La revisión de los 20 casos la prellenó el mismo modelo que extrajo; mide consistencia, no
   exactitud, hasta que una persona la confirme.